# Preprocessing & Visualization — Session 4
## The Titanic project — an advanced, nothing-hard-coded walkthrough

Microsoft Machine Learning Engineer Diploma

Sessions 1 to 3 gave you the three tools: NumPy for numbers, Pandas for tables, Matplotlib and Seaborn
for eyes. This session is the first time you use all three on **one real dataset, from the raw file to a
clean table you would be willing to hand to somebody else**.

---
**Scope — read this before you start**

This notebook stops where preprocessing stops. There is **no model in it** — no train/test split, no
fitting, no accuracy, no ROC curve. Those belong to the Machine Learning module, and doing them here
would teach you to reach for a model before you understand the table. Everything below is *preparation*
and *looking*, which is the part that decides whether any model later is worth anything.

---
**The rule this whole notebook is built on**

> **Nothing is hard-coded.**

You will not see a list of column names typed by hand, a threshold typed by hand, or a "drop these three
columns" line. Every column list is **derived from the frame**, every threshold is **computed from the
data**, and every decision is **printed with the number that justified it**.

That is not a style preference. A hard-coded notebook works once, on one file, on the day you wrote it.
Change the CSV, add a column, get a new month of data — and it breaks silently, which is the worst way
for anything to break. The code in here re-derives everything on every run, so it survives the data
changing under it.

---
**What makes this the *advanced* version**

- The dataset is **deliberately booby-trapped** — several columns are exact copies of each other under
  different names. Part 3 finds them **programmatically**, without you knowing the dataset.
- Missing data is treated with the mechanism analysis from the Missing Data guide, not a blind `fillna`.
- Outliers are measured per column *and* per group, and four different treatments are compared with numbers.
- Every chart is chosen by a rule the code applies, not by taste.
- Encoding and scaling are shown as what they are — **transformations of the table** — with a picture of
  what each one does to the shape of the data.

---
**The twelve parts**

| Part | What it answers |
|---|---|
| 1 | What is in this table, before I touch anything? |
| 2 | What does the whole table look like at once? |
| 3 | Are any of these columns the same column twice? |
| 4 | Where are the gaps, why are they there, and what do I do about each one? |
| 5 | Which values are extreme, and are they wrong or just rare? |
| 6 | Which columns actually relate to each other? |
| 7 | What new columns can I build from the ones I have? |
| 8 | How do I turn text and numbers into a consistent shape? |
| 9 | What does the cleaned table look like? |
| 10 | What is the one-page visual report? |
| 11 | What do I hand over, and how is it documented? |
| 12 | Your turn. |

---
# Part 1 — Load, and look before you touch

The single most common mistake in a data project is cleaning something before you know what it is.
Everything in this part is **read-only** — we do not change one value.

### 1.1 — Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.width', 110)

print('pandas    :', pd.__version__)
print('numpy     :', np.__version__)
print('seaborn   :', sns.__version__)

### 1.2 — Load the raw data, and keep a pristine copy

`raw` never gets touched again. Every change happens to `df`. You will be glad of this in Part 11, when
we work out what we actually changed.

In [ ]:
raw = sns.load_dataset('titanic')
df = raw.copy()          # everything we do happens to df; raw stays untouched

print('shape:', df.shape)
df.head()

### 1.3 — The three built-in looks

In [ ]:
df.info()

### 1.4 — `describe` on everything, not just the numbers

`describe()` alone shows only numeric columns. `include='all'` shows the text ones too, with a different
set of statistics — count, unique, top, freq.

In [ ]:
df.describe(include='all').T

### 1.5 — A machine-readable profile, instead of reading `info()` with your eyes

`info()` is for humans. A **frame** is for code: you can sort it, filter it, and write a rule against it.
That is the difference between an opinion and a decision.

In [ ]:
profile = pd.DataFrame({
    'dtype':     df.dtypes.astype(str),
    'n_missing': df.isna().sum(),
    'pct_missing': (df.isna().mean() * 100).round(1),
    'n_unique':  df.nunique(dropna=False),
    'pct_unique': (df.nunique(dropna=False) / len(df) * 100).round(1),
})
profile['sample'] = [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns]
profile = profile.sort_values('pct_missing', ascending=False)
profile

### 1.6 — The rules applied, still without naming anything

Four checks every table gets, written as thresholds rather than as observations.

In [ ]:
n_rows = len(df)

constant_cols = profile.index[profile['n_unique'] <= 1].tolist()
id_like_cols  = profile.index[profile['pct_unique'] > 95].tolist()
heavy_missing = profile.index[profile['pct_missing'] > 40].tolist()

print('rows                        :', n_rows)
print('duplicate rows              :', int(df.duplicated().sum()))
print('constant columns            :', constant_cols)
print('id-like columns (>95% uniq) :', id_like_cols)
print('heavily missing (>40%)      :', heavy_missing)

### 1.7 — Split the columns by *behaviour*, not by name

A numeric column with very few distinct values behaves like a category. `pclass` is stored as an integer,
but the distance between 1 and 2 is not the same kind of thing as the distance between 20 and 21 years of
age. The threshold below is a decision — but it is **one decision, written in one place**, instead of a
list of column names scattered through twenty cells.

In [ ]:
target = 'survived'      # the column the whole table is about; we describe it, we do not predict it

numeric_cols = df.select_dtypes(include='number').columns.drop(target, errors='ignore').tolist()
bool_cols    = df.select_dtypes(include='bool').columns.tolist()
cat_cols     = df.select_dtypes(include=['object', 'category', 'string', 'str']).columns.tolist()

low_card_numeric = [c for c in numeric_cols if df[c].nunique() <= 10]
true_numeric     = [c for c in numeric_cols if c not in low_card_numeric]

print('described by :', target)
print('true numeric :', true_numeric)
print('low-card num :', low_card_numeric)
print('boolean      :', bool_cols)
print('categorical  :', cat_cols)

### 1.8 — ⚠️ Fails on purpose: the summary that hides the problem

A reflex on any new table is `df.corr()`. Run it here and see what happens.

In [ ]:
# ⚠️ fails on purpose
try:
    print(df.corr())
except Exception as e:
    print(f'{type(e).__name__}: {e}')

The fix everyone reaches for is `numeric_only=True`. It works — but notice what it means: **most of this
table is not numeric**, so a correlation matrix is looking at a small corner of the data while feeling
like a summary of all of it. Part 6 measures the text columns properly.

In [ ]:
print(df.corr(numeric_only=True).round(3))
print()
print('columns the correlation matrix can see :', df.corr(numeric_only=True).shape[1])
print('columns in the table                   :', df.shape[1])

---
# Part 2 — Look at the whole table at once

Session 3 taught the charts. This part uses them as **instruments**: five figures that between them tell
you the shape of a table you have never seen, and every one of them is built from the derived lists in
Part 1 — so the same five cells work on any table you load.

### 2.1 — The missingness matrix

One pixel per cell of the table. Sorting the rows by a column with gaps makes any *pattern* in the
missingness visible immediately — and a pattern means the gaps are not random.

In [ ]:
gap_cols = df.columns[df.isna().any()].tolist()
print('columns with gaps:', gap_cols)

order_by = profile.index[profile['pct_missing'] > 0][0]      # the worst column, derived
view = df.sort_values(order_by, na_position='first')

fig, ax = plt.subplots(figsize=(9, 4))
ax.imshow(view.isna().to_numpy(), aspect='auto', cmap='Greys', interpolation='nearest')
ax.set_xticks(range(df.shape[1]))
ax.set_xticklabels(df.columns, rotation=90, fontsize=8)
ax.set_ylabel('rows (sorted by ' + order_by + ')')
ax.set_title('missingness matrix — black is missing')
plt.tight_layout(); plt.show()

### 2.2 — The same thing as a bar chart, with the numbers on it

In [ ]:
gaps = (df.isna().mean() * 100).sort_values(ascending=False)
gaps = gaps[gaps > 0]

fig, ax = plt.subplots(figsize=(6.4, 3))
bars = ax.barh(gaps.index[::-1], gaps.to_numpy()[::-1], color='#C0504D')
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=9)
ax.set_xlim(0, gaps.max() * 1.25)
ax.set_xlabel('% missing')
ax.set_title('how much is missing, per column')
plt.tight_layout(); plt.show()

### 2.3 — Do the gaps travel together?

This is the question the bar chart cannot answer. If two columns are always missing on the *same rows*,
they are missing for one shared reason — and you treat them as one problem, not two.

In [ ]:
flags = df[gap_cols].isna().astype(int)

if flags.shape[1] > 1 and flags.nunique().min() > 1:
    nullity = flags.corr()
    fig, ax = plt.subplots(figsize=(4.4, 3.4))
    sns.heatmap(nullity, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, ax=ax)
    ax.set_title('do these columns go missing together?')
    plt.tight_layout(); plt.show()
    print(nullity.round(3).to_string())
else:
    print('not enough varying gap columns to correlate')

### 2.4 — Every numeric column, in one figure

The loop is over `true_numeric` — the list Part 1 derived. Add a column to the data and this cell draws
it too, with no edit.

In [ ]:
show = true_numeric + low_card_numeric
ncol = 3
nrow = int(np.ceil(len(show) / ncol))

fig, axs = plt.subplots(nrow, ncol, figsize=(4 * ncol, 2.7 * nrow))
axs = np.atleast_1d(axs).ravel()
for ax, col in zip(axs, show):
    s = df[col].dropna()
    bins = min(30, max(5, s.nunique()))
    ax.hist(s, bins=bins, color='#2E74B5', edgecolor='white')
    ax.set_title(f'{col}   skew {s.skew():.2f}', fontsize=10)
for ax in axs[len(show):]:
    ax.set_visible(False)
fig.suptitle('every numeric column, drawn by the same loop', y=1.01, fontsize=13)
plt.tight_layout(); plt.show()

### 2.5 — Every categorical column, in one figure

Same idea, different chart — because the *type* of the column chooses the chart, not you. Note the
`order=` argument: an unordered category read in a random order is much harder to read than the same
one sorted by size.

In [ ]:
show_cat = [c for c in cat_cols + bool_cols if df[c].nunique(dropna=False) <= 12]
ncol = 3
nrow = int(np.ceil(len(show_cat) / ncol))

fig, axs = plt.subplots(nrow, ncol, figsize=(4 * ncol, 2.7 * nrow))
axs = np.atleast_1d(axs).ravel()
for ax, col in zip(axs, show_cat):
    counts = df[col].value_counts(dropna=False)
    ax.bar([str(i) for i in counts.index], counts.to_numpy(), color='#548235')
    ax.set_title(f'{col}   {df[col].nunique(dropna=False)} levels', fontsize=10)
    ax.tick_params(axis='x', rotation=30, labelsize=8)
for ax in axs[len(show_cat):]:
    ax.set_visible(False)
fig.suptitle('every categorical column, drawn by the same loop', y=1.01, fontsize=13)
plt.tight_layout(); plt.show()

### 2.6 — The column the table is about

Whatever question the table exists to answer, look at that column on its own before anything else. Here
it is `survived`. We are **describing** it, not predicting it.

In [ ]:
counts = df[target].value_counts().sort_index()
shares = df[target].value_counts(normalize=True).sort_index()
summary = pd.DataFrame({'count': counts, 'share': shares.round(4)})
print(summary)
print()
print('minority share:', round(shares.min(), 4))

fig, ax = plt.subplots(figsize=(4.6, 3))
sns.countplot(x=target, data=df, ax=ax)
ax.bar_label(ax.containers[0])
ax.set_title(f'{target} — {shares.min():.1%} in the smaller group')
plt.show()

### 2.7 — The imbalance is a preprocessing fact, not a modelling one

Why it matters *here*: a group that is 5% of the table will be 5% of any sample you take, 5% of any
chart you draw, and can vanish entirely from a small subset. That changes how you split, sample and plot
long before anyone trains anything.

In [ ]:
minority_share = shares.min()

if minority_share < 0.05:
    verdict = 'severe imbalance -> stratify every sample, and never eyeball a random subset'
elif minority_share < 0.25:
    verdict = 'moderate imbalance -> stratify samples, and always plot rates not counts'
else:
    verdict = 'roughly balanced -> counts and rates tell the same story'

print(f'minority share = {minority_share:.3f}')
print('verdict:', verdict)

---
# Part 3 — Is any column the same column twice?

This table has copies hidden in it. Not obvious copies — different names, different dtypes, one of them
text and one of them a number. We are going to find them **without knowing the dataset**, because on your
own data you will not know it either.

### 3.1 — Redundancy between numeric columns

The classic tool, used properly: take the upper triangle so each pair is counted once and the diagonal
(a column with itself, always 1.0) does not sit at the top of the list.

In [ ]:
num_for_corr = df.select_dtypes(include='number')
c = num_for_corr.corr().abs()

pairs = (c.where(np.triu(np.ones(c.shape), k=1).astype(bool))
          .stack()
          .sort_values(ascending=False))

print('most correlated numeric pairs:')
print(pairs.head(8).round(3).to_string())
print()
print('pairs above 0.9 (near-duplicates):', pairs[pairs > 0.9].index.tolist())

The answer is: none. On the numeric side this table looks clean — which is exactly why the next test
matters. Correlation can only see numbers, and most of this table is text.

### 3.2 — Redundancy between *any* two columns: the one-to-one test

The idea in one line: **if every value of A maps to exactly one value of B, then A determines B.** No
correlation, no dtype requirement, works on text and numbers alike.

In [ ]:
from itertools import combinations

check_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 20]

links = []
for a, b in combinations(check_cols, 2):
    sub = df[[a, b]].dropna()
    if len(sub) == 0:
        continue
    a_to_b = sub.groupby(a, observed=True)[b].nunique().max()
    b_to_a = sub.groupby(b, observed=True)[a].nunique().max()
    if a_to_b == 1 or b_to_a == 1:
        links.append({
            'a': a, 'b': b,
            'a_determines_b': a_to_b == 1,
            'b_determines_a': b_to_a == 1,
            'one_to_one': a_to_b == 1 and b_to_a == 1,
        })

linkage = pd.DataFrame(links)
print(f'{len(linkage)} determined pairs found among {len(check_cols)} low-cardinality columns')
linkage

### 3.3 — Proof, without a model: the crosstab

A one-to-one pair produces a crosstab with exactly one non-zero cell per row and per column. That is what
"the same column twice" looks like on paper.

In [ ]:
one_to_one = linkage[linkage['one_to_one']]

for _, r in one_to_one.iterrows():
    a, b = r['a'], r['b']
    print('=' * 60)
    print(f'{a}  vs  {b}')
    print(pd.crosstab(df[a], df[b]).to_string())
    print()

Read the first table. Every row has one number and the rest zeros — so knowing one column tells you the
other with no uncertainty at all. Keeping both would mean carrying the same information twice under two
names, and every summary, chart and correlation you produce afterwards would double-count it.

### 3.4 — And a picture of the same fact

In [ ]:
pair = one_to_one.iloc[0]
a, b = pair['a'], pair['b']

fig, ax = plt.subplots(figsize=(4.2, 3.2))
sns.heatmap(pd.crosstab(df[a], df[b]), annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
ax.set_title(f'{a} x {b} — one filled cell per row means one column, twice')
plt.tight_layout(); plt.show()

### 3.5 — Drop the redundant columns — by rule, not by name

Three rules, all computed. First: **never drop the column the table is about** — if a pair contains it,
the other one goes. Then the tie-breakers for everything else: keep the column with fewer missing values;
if they tie, keep the shorter name. You will get the same answer tomorrow on a different table without
editing anything.

In [ ]:
to_drop = set()

if len(linkage):
    for _, r in linkage[linkage['one_to_one']].iterrows():
        a, b = r['a'], r['b']
        if a in to_drop or b in to_drop:
            continue
        if target in (a, b):
            loser = b if a == target else a      # never drop the column the table is about
        else:
            loser = b if (df[a].isna().sum(), len(a)) <= (df[b].isna().sum(), len(b)) else a
        to_drop.add(loser)

to_drop |= set(constant_cols) | set(id_like_cols)
to_drop.discard(target)
to_drop = sorted(to_drop)

print('dropping:', to_drop)
df = df.drop(columns=to_drop)
print('shape now:', df.shape)

### 3.6 — One-way dependence is not the same as duplication

Look back at the linkage table: one pair had `a_determines_b` true but `b_determines_a` false. That is not
a copy — it is a **hierarchy**, one column being a coarser version of the other. You usually keep both,
because the coarse one is easier to plot and the fine one carries more information.

In [ ]:
one_way = linkage[linkage['one_to_one'] == False] if len(linkage) else pd.DataFrame()

for _, r in one_way.iterrows():
    a, b = r['a'], r['b']
    if a not in df.columns or b not in df.columns:
        continue
    fine, coarse = (a, b) if r['a_determines_b'] else (b, a)
    print(f'{fine} determines {coarse}, but not the other way round:')
    print(pd.crosstab(df[fine], df[coarse]).to_string())
    print()

### 3.7 — Re-derive the column lists, because the frame changed

This is the line that separates a notebook that survives from one that breaks in the middle. Any list
computed **before** a structural change is now stale.

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.drop(target, errors='ignore').tolist()
bool_cols    = df.select_dtypes(include='bool').columns.tolist()
cat_cols     = df.select_dtypes(include=['object', 'category', 'string', 'str']).columns.tolist()
low_card_numeric = [c for c in numeric_cols if df[c].nunique() <= 10]
true_numeric     = [c for c in numeric_cols if c not in low_card_numeric]

print('true numeric :', true_numeric)
print('low-card num :', low_card_numeric)
print('boolean      :', bool_cols)
print('categorical  :', cat_cols)

---
# Part 4 — Missing data: the mechanism, not the `fillna`

The Missing Data study guide from Session 2 is the reference for this part. Here it gets applied to a real
table, with the tests run and the numbers printed.

### 4.1 — ⚠️ The gotcha that eats an hour

This is the line you will find in every tutorial on the internet. Watch the counter.

In [ ]:
import warnings

num_with_gaps = [c for c in df.columns
                 if df[c].isna().any() and pd.api.types.is_numeric_dtype(df[c])]
col = num_with_gaps[0]

test = df.copy()
before = int(test[col].isna().sum())

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    test[col].fillna(test[col].median(), inplace=True)   # the line you see everywhere

for w in caught:
    print(f'{w.category.__name__}: {str(w.message).splitlines()[0]}')

print()
print(f'column        : {col}')
print('missing before:', before)
print('missing after :', int(test[col].isna().sum()))
print('did anything happen?', before != int(test[col].isna().sum()))

The code ran, raised no error, and did **nothing**. `test[col]` hands you a Series that may be a copy;
`inplace=True` fills the copy, and the copy is thrown away. In pandas 3 you at least get a warning. In
older versions it passed in complete silence.

> **The correct form is always `df[col] = df[col].fillna(value)`.**

In [ ]:
test[col] = test[col].fillna(test[col].median())
print('missing after the correct form:', int(test[col].isna().sum()))

### 4.2 — Where are the gaps, and how much do they cost?

In [ ]:
gaps = pd.DataFrame({
    'n_missing': df.isna().sum(),
    'pct': (df.isna().mean() * 100).round(2),
})
gaps = gaps[gaps['n_missing'] > 0].sort_values('pct', ascending=False)
print(gaps)
print()
print('rows with at least one gap:', int(df.isna().any(axis=1).sum()), 'of', len(df))
print('rows surviving dropna()   :', len(df.dropna()))
print('cost of listwise deletion :', f'{1 - len(df.dropna())/len(df):.1%} of the data')

### 4.3 — Which mechanism? Compare the two groups, per column

For each column with gaps: split the rows into *missing here* and *present here*, then compare every other
column between the two groups. A significant difference means the gaps are **related to something you can
see** — MAR, not MCAR.

In [ ]:
from scipy import stats

missing_cols = gaps.index.tolist()
others = [c for c in df.columns]

for col in missing_cols:
    flag = df[col].isna()
    print('=' * 68)
    print(f'{col}  —  {flag.mean():.1%} missing')
    rows = []
    for o in others:
        if o == col:
            continue
        s = df[o]
        if pd.api.types.is_numeric_dtype(s) or pd.api.types.is_bool_dtype(s):
            a, b = s[flag].astype(float).dropna(), s[~flag].astype(float).dropna()
            if len(a) > 1 and len(b) > 1 and a.var() > 0 and b.var() > 0:
                t, p = stats.ttest_ind(a, b, equal_var=False)
                rows.append({'other': o, 'test': 't', 'when_missing': round(a.mean(), 2),
                             'when_present': round(b.mean(), 2), 'p': p})
        else:
            tab = pd.crosstab(s, flag)
            if tab.shape[1] == 2 and tab.shape[0] > 1 and tab.to_numpy().min() >= 0:
                chi2, p, _, _ = stats.chi2_contingency(tab)
                rows.append({'other': o, 'test': 'chi2', 'when_missing': np.nan,
                             'when_present': np.nan, 'p': p})
    rep = pd.DataFrame(rows)
    rep['p'] = rep['p'].astype(float)
    rep = rep.sort_values('p')
    rep['significant'] = rep['p'] < 0.05
    print(rep.head(6).to_string(index=False, float_format=lambda v: f'{v:.4g}'))
    print()

### 4.4 — And the same finding as a picture

The table proves it; the figure makes it obvious. For the worst column, draw the distribution of a few
other columns split by whether this one is missing.

In [ ]:
worst_gap = gaps.index[0]
flag = df[worst_gap].isna()

compare = [c for c in true_numeric + low_card_numeric if c in df.columns][:3]
fig, axs = plt.subplots(1, len(compare), figsize=(4 * len(compare), 3))
for ax, c in zip(np.atleast_1d(axs), compare):
    sns.kdeplot(df.loc[flag, c].dropna(), label=f'{worst_gap} missing', fill=True, ax=ax)
    sns.kdeplot(df.loc[~flag, c].dropna(), label=f'{worst_gap} present', fill=True, ax=ax)
    ax.set_title(c); ax.legend(fontsize=8)
fig.suptitle(f'the rows where {worst_gap} is missing are not a random sample', y=1.03, fontsize=12)
plt.tight_layout(); plt.show()

If those curves sat on top of each other, the gaps would be MCAR and you could delete them without bias.
They do not, so you cannot.

### 4.5 — Treat each column according to its own verdict

Three rules, applied by threshold. No column names.

In [ ]:
plan = {}
for col in missing_cols:
    pct = df[col].isna().mean()
    if pct > 0.4:
        plan[col] = 'indicator + drop'
    elif pd.api.types.is_numeric_dtype(df[col]):
        plan[col] = 'group-wise median'
    else:
        plan[col] = 'mode' if pct < 0.02 else 'own category'

for k, v in plan.items():
    print(f'{k:<14} {df[k].isna().mean():6.1%}  ->  {v}')

### 4.6 — Heavily-missing columns: keep the *fact*, drop the values

77% missing is too little to impute honestly. But **whether the value was recorded at all** is itself
information, and it costs one column to keep.

In [ ]:
for col, action in plan.items():
    if action == 'indicator + drop':
        df[col + '_known'] = df[col].notna().astype(int)
        rate = df.groupby(col + '_known', observed=True)[target].mean().round(3)
        print(f'{col}: {target} rate by whether it is known ->')
        print(rate.to_string())
        df = df.drop(columns=[col])
        print()

### 4.7 — Group-wise imputation, and what it buys over the global median

The global median puts one number into every gap, which builds an artificial spike in the middle of the
distribution. A median computed **within a group** puts a different number into each gap, so the natural
spread survives.

In [ ]:
group_keys = [c for c in cat_cols + low_card_numeric if c in df.columns and df[c].isna().sum() == 0]
group_keys = group_keys[:2]     # two keys keeps the groups big enough to be stable
print('grouping numeric imputation by:', group_keys)
print()

before_fill = {}
for col, action in plan.items():
    if action != 'group-wise median' or col not in df.columns:
        continue
    before_fill[col] = df[col].copy()
    truth_mean = df[col].mean()
    global_filled = df[col].fillna(df[col].median())
    grouped = df.groupby(group_keys, observed=True)[col].transform('median')
    group_filled = df[col].fillna(grouped).fillna(df[col].median())

    print(f'--- {col} ---')
    print(f'observed mean (before filling) : {truth_mean:8.3f}')
    print(f'after global median fill       : {global_filled.mean():8.3f}   std {global_filled.std():7.3f}')
    print(f'after group-wise median fill   : {group_filled.mean():8.3f}   std {group_filled.std():7.3f}')
    df[col + '_was_missing'] = df[col].isna().astype(int)
    df[col] = group_filled
    print()

### 4.8 — The picture that makes the difference undeniable

In [ ]:
col = list(before_fill)[0]
original = before_fill[col]
global_v = original.fillna(original.median())

fig, axs = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, (s, name, colour) in zip(axs, [
        (original.dropna(), 'observed only', '#666666'),
        (global_v,          'global median fill', '#C0504D'),
        (df[col],           'group-wise fill', '#548235')]):
    ax.hist(s, bins=30, color=colour, edgecolor='white')
    ax.set_title(f'{name}\nstd {s.std():.2f}', fontsize=10)
fig.suptitle(f'{col} — what each strategy does to the shape', y=1.04, fontsize=12)
plt.tight_layout(); plt.show()

Look at the middle panel: a spike that is not in the data, only in the filling. The right panel keeps the
shape because it uses a different number for each group.

### 4.9 — The small categorical gaps

In [ ]:
for col, action in plan.items():
    if col not in df.columns:
        continue
    if action == 'mode':
        fill = df[col].mode()[0]
        print(f'{col}: {int(df[col].isna().sum())} rows -> mode "{fill}"')
        df[col] = df[col].fillna(fill)
    elif action == 'own category':
        print(f"{col}: {int(df[col].isna().sum())} rows -> new category 'Missing'")
        if isinstance(df[col].dtype, pd.CategoricalDtype):
            df[col] = df[col].cat.add_categories(['Missing'])
        df[col] = df[col].fillna('Missing')

print()
print('missing values left in the frame:', int(df.isna().sum().sum()))

### 4.10 — ✍️ Your turn

Re-run the group-wise imputation for the numeric column, but group by **three** keys instead of two.
Print how many groups you end up with and how many of them have fewer than 10 rows. Then say in one
sentence why more grouping keys is not automatically better.

In [ ]:
# your code here


---
# Part 5 — Outliers and skew, measured

Session 3 taught the box plot and the IQR rule. Here they get applied to every numeric column at once,
and — more importantly — the question after the rule: **is this value wrong, or just rare?**

### 5.1 — Scan every true-numeric column at once

In [ ]:
out_rows = []
for col in [c for c in true_numeric if c in df.columns]:
    s = df[col]
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flagged = ((s < lo) | (s > hi)).sum()
    out_rows.append({'column': col, 'Q1': round(q1, 2), 'Q3': round(q3, 2),
                     'low_fence': round(lo, 2), 'high_fence': round(hi, 2),
                     'n_outliers': int(flagged), 'pct': round(flagged / len(s) * 100, 1),
                     'skew': round(float(s.skew()), 2)})

outliers = pd.DataFrame(out_rows).sort_values('pct', ascending=False)
outliers

Notice the negative low fence. That is not a bug — it means the rule cannot flag anything on the left,
because the smallest possible value is zero. The arithmetic does not know that prices are never negative.

### 5.2 — Look at the worst offender

In [ ]:
worst = outliers.iloc[0]['column']
s = df[worst]

fig, axs = plt.subplots(1, 3, figsize=(12, 3))
axs[0].hist(s, bins=40, color='#2E74B5', edgecolor='white')
axs[0].set_title(f'{worst} — raw (skew {s.skew():.2f})')
sns.boxplot(x=s, ax=axs[1]); axs[1].set_title('the IQR view')
axs[2].hist(np.log1p(s), bins=40, color='#548235', edgecolor='white')
axs[2].set_title(f'log1p({worst}) — skew {np.log1p(s).skew():.2f}')
plt.tight_layout(); plt.show()

### 5.3 — Four treatments, compared on one column

There is no correct answer here, only a trade-off you have to see. Every row of this table is a real
option, and the columns tell you what each one costs.

In [ ]:
q1, q3 = s.quantile([0.25, 0.75]); iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
mask = (s < lo) | (s > hi)

comparison = pd.DataFrame({
    'keep everything': [len(s), s.mean(), s.std(), s.skew(), s.max()],
    'drop outliers':   [int((~mask).sum()), s[~mask].mean(), s[~mask].std(), s[~mask].skew(), s[~mask].max()],
    'winsorise (clip)':[len(s), s.clip(lo, hi).mean(), s.clip(lo, hi).std(), s.clip(lo, hi).skew(), s.clip(lo, hi).max()],
    'log1p':           [len(s), np.log1p(s).mean(), np.log1p(s).std(), np.log1p(s).skew(), np.log1p(s).max()],
}, index=['n', 'mean', 'std', 'skew', 'max']).T.round(3)
comparison

### 5.4 — And the same four, drawn

In [ ]:
variants = {
    'keep everything': s,
    'drop outliers':   s[~mask],
    'winsorise (clip)': s.clip(lo, hi),
    'log1p':           np.log1p(s),
}

fig, axs = plt.subplots(1, 4, figsize=(14, 2.9))
for ax, (name, v) in zip(axs, variants.items()):
    ax.hist(v, bins=35, color='#7C5CBF', edgecolor='white')
    ax.set_title(f'{name}\nn={len(v)}  skew={v.skew():.2f}', fontsize=10)
fig.suptitle(f'{worst} — four ways to deal with the same tail', y=1.06, fontsize=12)
plt.tight_layout(); plt.show()

One of them loses rows — and the rows it loses are the most expensive passengers, who are exactly the
ones a report about fares should not be missing. One of them fixes the shape and keeps every row. That is
the argument for the log, and it is visible, not theoretical.

### 5.5 — Outliers are relative to a group

A fare that is extreme for the whole ship may be perfectly ordinary inside first class. Compute the fence
**per group** and the number changes.

In [ ]:
group_col = cat_cols[0] if cat_cols else low_card_numeric[0]

rows = []
for g, part in df.groupby(group_col, observed=True):
    v = part[worst]
    a, b = v.quantile([0.25, 0.75]); i = b - a
    rows.append({group_col: g, 'n': len(v), 'Q1': round(a, 1), 'Q3': round(b, 1),
                 'high_fence': round(b + 1.5 * i, 1),
                 'n_outliers': int(((v < a - 1.5*i) | (v > b + 1.5*i)).sum())})

print(pd.DataFrame(rows).to_string(index=False))
print()
print('outliers on the whole column :', int(mask.sum()))
print('outliers summed within groups:', sum(r['n_outliers'] for r in rows))

fig, ax = plt.subplots(figsize=(6.4, 3.2))
sns.boxplot(x=group_col, y=worst, data=df, ax=ax)
ax.set_title(f'{worst} by {group_col} — the fence moves per group')
plt.show()

### 5.6 — The decision, written down

We keep every row and add a transformed copy of the skewed column. Nothing is deleted, because none of
these values is impossible — they are rare, and rare is data.

In [ ]:
skew_threshold = 1.0
to_transform = outliers.loc[outliers['skew'].abs() > skew_threshold, 'column'].tolist()
print(f'columns with |skew| > {skew_threshold}:', to_transform)

for col in to_transform:
    if (df[col] < 0).any():
        print(f'{col}: has negative values, log1p not applicable — skipped')
        continue
    df[col + '_log'] = np.log1p(df[col])
    print(f"{col}: added {col}_log   skew {df[col].skew():.2f} -> {df[col + '_log'].skew():.2f}")

---
# Part 6 — Which columns actually relate to each other?

The correlation matrix in Part 1 could only see the numeric columns. This part measures **every** column
against the one the table is about, choosing the right statistic for the dtype — and it does it with
statistics, not with a model.

### 6.1 — The right test per column type

Two measures, both **effect sizes** — that is, both answer "how big is the difference", not "is it real".
On 891 rows almost anything is statistically significant, so ranking by p-value would be misleading.

- **Cramér's V** for two categorical columns — 0 means unrelated, 1 means one determines the other.
- **Cohen's d** for a numeric column split by a categorical one — the gap between the two means, measured
  in standard deviations.

In [ ]:
assoc = []
for col in [c for c in df.columns if c != target]:
    s = df[col]
    if pd.api.types.is_bool_dtype(s) or s.nunique() <= 12:
        tab = pd.crosstab(s, df[target])
        if tab.shape[0] > 1 and tab.shape[1] > 1:
            chi2, p, _, _ = stats.chi2_contingency(tab)
            n = tab.to_numpy().sum()
            cramers_v = np.sqrt(chi2 / (n * (min(tab.shape) - 1)))
            assoc.append({'column': col, 'test': 'chi2', 'strength': round(cramers_v, 3), 'p': p})
    elif pd.api.types.is_numeric_dtype(s):
        a = s[df[target] == 1].dropna()
        b = s[df[target] == 0].dropna()
        t, p = stats.ttest_ind(a, b, equal_var=False)
        pooled = np.sqrt((a.var() + b.var()) / 2)
        d = abs(a.mean() - b.mean()) / pooled if pooled else 0.0
        assoc.append({'column': col, 'test': 't', 'strength': round(float(d), 3), 'p': p})

assoc_df = pd.DataFrame(assoc).sort_values('strength', ascending=False)
assoc_df['significant'] = assoc_df['p'] < 0.05
assoc_df.round({'p': 6})

### 6.2 — A second opinion: the correlation ratio

Cohen's d compares two means and nothing else, so a relationship that is strong at the *edges* and flat in
the middle can score near zero. The correlation ratio (eta squared) asks a different question: **how much
of the spread in this numeric column is explained by which group a row belongs to?** It catches
non-linear structure that a difference of means misses.

In [ ]:
def correlation_ratio(values, groups):
    """eta squared — share of variance in `values` explained by `groups`."""
    ok = values.notna()
    v, g = values[ok], groups[ok]
    grand = v.mean()
    between = sum(len(v[g == k]) * (v[g == k].mean() - grand) ** 2 for k in g.unique())
    total = ((v - grand) ** 2).sum()
    return between / total if total else 0.0

eta_rows = []
for col in [c for c in df.columns if c != target and pd.api.types.is_numeric_dtype(df[c])]:
    eta_rows.append({'column': col,
                     'eta_sq_by_target': round(correlation_ratio(df[col], df[target]), 4)})

eta_df = pd.DataFrame(eta_rows).sort_values('eta_sq_by_target', ascending=False)
eta_df

### 6.3 — Put the two side by side

Where the two rankings agree, you can be confident. Where they disagree, the disagreement is the
interesting part — it usually means the relationship is not a simple shift in the average.

In [ ]:
ranks = assoc_df[['column', 'strength']].merge(eta_df, on='column', how='outer')

measures = ['strength', 'eta_sq_by_target']
for c in measures:
    ranks[c + '_rank'] = ranks[c].rank(ascending=False)

# a text column has no correlation ratio, so say how many measures each row was ranked on
ranks['n_measures'] = ranks[[c + '_rank' for c in measures]].notna().sum(axis=1)
ranks['mean_rank']  = ranks[[c + '_rank' for c in measures]].mean(axis=1)
ranks = ranks.sort_values(['mean_rank', 'strength'], ascending=[True, False])

print(ranks[['column', 'strength', 'eta_sq_by_target', 'n_measures', 'mean_rank']].to_string(index=False))

Read `n_measures` before you read `mean_rank`. A text column has no correlation ratio — the ratio needs a
numeric column to measure the spread of — so those rows are ranked on one measure only. A rank averaged
over one number and a rank averaged over two are not the same kind of evidence, and a table that hides
that difference is a table that will mislead you.

### 6.4 — The relationships, drawn

The code picks the chart from the dtype: a categorical column gets a rate bar chart with the overall rate
as a reference line; a continuous one gets two distributions.

In [ ]:
top_cols = ranks['column'].head(4).tolist()
print('drawing the top 4 by mean rank:', top_cols)

fig, axs = plt.subplots(2, 2, figsize=(11, 6.5))
for ax, col in zip(axs.flat, top_cols):
    s = df[col]
    if pd.api.types.is_numeric_dtype(s) and s.nunique() > 10:
        sns.kdeplot(data=df, x=col, hue=target, fill=True, common_norm=False, ax=ax)
        ax.set_title(f'{col} — distribution by {target}')
    else:
        sns.barplot(x=col, y=target, data=df, ax=ax, errorbar=('ci', 95))
        ax.axhline(df[target].mean(), color='gray', linestyle='--', linewidth=1)
        ax.set_title(f'{col} — {target} rate (dashed = overall)')
        ax.tick_params(axis='x', rotation=20)
fig.suptitle('the four strongest relationships in the table', y=1.01, fontsize=13)
plt.tight_layout(); plt.show()

### 6.5 — And the relationships *between* the other columns

Not everything has to be measured against the target. Two columns that carry the same information will
make every later summary double-count it — this is the multicollinearity check, done visually.

In [ ]:
num_now = df.select_dtypes(include='number')
corr = num_now.corr()

mask_up = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(7.5, 5.5))
sns.heatmap(corr, mask=mask_up, annot=True, fmt='.2f', cmap='RdBu_r',
            vmin=-1, vmax=1, ax=ax, annot_kws={'size': 7})
ax.set_title('numeric columns against each other (lower triangle only)')
plt.tight_layout(); plt.show()

---
# Part 7 — Feature engineering that is derived, not invented

A new column has to earn its place. The rule in this part: **build it from a rule, then defend it with a
number and a picture.** If you cannot show that it separates anything, it does not go in.

### 7.1 — Combine the columns that measure the same thing

The code does not look for `sibsp` and `parch` by name. It looks for **count-like columns**: integer,
never negative, few distinct values.

In [ ]:
count_like = [c for c in df.columns
              if c != target
              and pd.api.types.is_integer_dtype(df[c])
              and df[c].min() >= 0
              and df[c].nunique() <= 12
              and not str(df[c].dtype) == 'bool'
              and not c.endswith('_known')
              and not c.endswith('_was_missing')]
print('count-like columns:', count_like)

if len(count_like) >= 2:
    df['family_size'] = df[count_like].sum(axis=1) + 1      # +1 = the passenger
    df['is_alone'] = (df['family_size'] == 1).astype(int)
    print()
    print(df.groupby('family_size', observed=True)[target]
            .agg(['size', 'mean']).round(3).to_string())

### 7.2 — And the shape it makes

Read the numbers above, then look at the picture. Neither of the two original columns shows this on its
own — the sum does.

In [ ]:
fam = df.groupby('family_size', observed=True)[target].agg(['size', 'mean'])

fig, ax1 = plt.subplots(figsize=(7, 3.4))
ax1.bar(fam.index, fam['size'], color='#D9E2EC')
ax1.set_ylabel('passengers', color='#7B8794')
ax1.set_xlabel('family_size')

ax2 = ax1.twinx()
ax2.plot(fam.index, fam['mean'], marker='o', color='#C0504D', linewidth=2)
ax2.axhline(df[target].mean(), color='gray', linestyle='--', linewidth=1)
ax2.set_ylabel(f'{target} rate', color='#C0504D')
ax2.set_ylim(0, 1)
ax1.set_title('family_size — bars are how many, the line is the rate')
plt.tight_layout(); plt.show()

The line goes up and then falls off a cliff. That is not something a straight line can describe, which is
the whole reason the next cell puts it into bands.

### 7.3 — Bins that come from the data, not from your intuition

Two different tools that people confuse constantly:

- `cut` splits on **values** you choose — use it when the boundaries mean something.
- `qcut` splits on **counts**, so every bin has the same number of rows — use it when you have no idea
  where the boundaries should be.

In [ ]:
edges = [0, 1, 4, df['family_size'].max()]
labels = ['alone', 'small', 'large']
df['family_band'] = pd.cut(df['family_size'], bins=edges, labels=labels, include_lowest=True)

print('cut — boundaries chosen for meaning:')
print(df.groupby('family_band', observed=True)[target].agg(['size', 'mean']).round(3).to_string())
print()

df[worst + '_band'] = pd.qcut(df[worst], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
print('qcut — equal counts per bin:')
print(df.groupby(worst + '_band', observed=True)[target].agg(['size', 'mean']).round(3).to_string())

### 7.4 — The two side by side

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.2))
for ax, col, name in zip(axs, ['family_band', worst + '_band'], ['cut (by value)', 'qcut (by count)']):
    g = df.groupby(col, observed=True)[target].agg(['size', 'mean'])
    bars = ax.bar([str(i) for i in g.index], g['mean'], color='#2AA198')
    ax.bar_label(bars, fmt='%.3f', padding=2, fontsize=9)
    for i, n in enumerate(g['size']):
        ax.text(i, 0.02, f'n={n}', ha='center', fontsize=8, color='white')
    ax.axhline(df[target].mean(), color='gray', linestyle='--', linewidth=1)
    ax.set_ylim(0, 1)
    ax.set_title(name)
plt.tight_layout(); plt.show()

### 7.5 — Ratios and transforms

A ticket price is for the ticket, not for the person — so a family of five on one ticket looks enormously
expensive per head unless you divide. This is not a clever trick, it is **undoing an artefact of how the
data was recorded**.

In [ ]:
if 'family_size' in df.columns:
    df[worst + '_per_person'] = df[worst] / df['family_size']

new_cols = [c for c in [worst + '_log', worst + '_per_person'] if c in df.columns]
print(df[[worst] + new_cols].describe().round(2).to_string())
print()
for c in new_cols:
    a = df.loc[df[target] == 1, c]
    b = df.loc[df[target] == 0, c]
    t, p = stats.ttest_ind(a, b, equal_var=False)
    print(f'{c:<22} group 1 {a.mean():8.3f}   group 0 {b.mean():8.3f}   p = {p:.3g}')

### 7.6 — Interactions: the effect that only exists inside a group

Some relationships are not additive. If a column behaves one way inside one group and the opposite way
inside another, no summary that treats the two columns separately will ever show it. The fix is to make
the **combination** its own column.

In [ ]:
cat_for_inter = [c for c in df.columns
                 if c != target and df[c].nunique() <= 4
                 and not pd.api.types.is_float_dtype(df[c])
                 and not c.endswith('_band')][:2]
print('interacting:', cat_for_inter)

if len(cat_for_inter) == 2:
    a, b = cat_for_inter
    pivot = df.pivot_table(index=a, columns=b, values=target, aggfunc='mean', observed=True)
    print()
    print(pivot.round(3).to_string())

    fig, ax = plt.subplots(figsize=(5.2, 3.2))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1, ax=ax)
    ax.set_title(f'{target} rate by {a} x {b}')
    plt.tight_layout(); plt.show()

    df[f'{a}_x_{b}'] = df[a].astype(str) + '_' + df[b].astype(str)
    print()
    print('new interaction column:', f'{a}_x_{b}', '->', df[f'{a}_x_{b}'].nunique(), 'levels')

### 7.7 — ✍️ Your turn

Build one more feature of your own and defend it with a number. Suggestions: a flag for whether the fare
was zero, or the passenger's fare as a percentile **within their own class**. Print the rate by your new
feature and a p-value, then say in one sentence whether you would keep it.

In [ ]:
# your code here


---
# Part 8 — Encoding and scaling, as transformations of the table

These two words get used as if they were magic. They are not: they are **changes to the shape and the
units of your table**, and you can see exactly what each one does.

Everything here is done in pandas and NumPy — the tools you already have.

### 8.1 — One-hot encoding: text becomes columns

The column list is derived, and `drop_first` matters: with two levels, the second column is just `1 -` the
first, so it carries no new information and only creates a perfectly redundant pair.

In [ ]:
encode_cols = [c for c in df.columns
               if (not pd.api.types.is_numeric_dtype(df[c])) and not c.endswith('_band')]
print('columns to encode :', encode_cols)
print('levels in each    :', {c: int(df[c].nunique()) for c in encode_cols})
print()

encoded = pd.get_dummies(df[encode_cols], drop_first=True, dtype=int)
print('columns in  :', len(encode_cols))
print('columns out :', encoded.shape[1])
print()
print(encoded.head().to_string())

### 8.2 — What it did, in a picture

One text column with three levels becomes two zero/one columns. The third level is not lost — it is the
row where both of the new columns are zero.

In [ ]:
demo_col = max(encode_cols, key=lambda c: df[c].nunique() if df[c].nunique() <= 5 else 0)
demo = pd.get_dummies(df[[demo_col]], drop_first=True, dtype=int).head(8)

fig, axs = plt.subplots(1, 2, figsize=(9, 2.8),
                        gridspec_kw={'width_ratios': [1, demo.shape[1]]})
axs[0].axis('off')
axs[0].set_title(f'before — {demo_col}', fontsize=10)
for i, v in enumerate(df[demo_col].head(8)):
    axs[0].text(0.5, 1 - (i + 0.5) / 8, str(v), ha='center', va='center', fontsize=9)
sns.heatmap(demo, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axs[1])
axs[1].set_title('after — one column per level, minus one', fontsize=10)
plt.tight_layout(); plt.show()

### 8.3 — ⚠️ The encoding mistake that changes your data's meaning

Replacing text with 1, 2, 3 is quick and, for an **unordered** column, wrong: it tells everything
downstream that the third level is three times the first, and that the gap between levels 1 and 2 equals
the gap between 2 and 3. Neither is true for a port of departure.

In [ ]:
unordered = [c for c in encode_cols if df[c].nunique() == 3][0]
codes = {v: i + 1 for i, v in enumerate(sorted(df[unordered].astype(str).unique()))}
print(f'a naive numeric code for {unordered}: {codes}')
print()
print('this now claims:')
keys = list(codes)
print(f'   {keys[2]} - {keys[1]} == {keys[1]} - {keys[0]}   (equal spacing)')
print(f'   {keys[2]} == 3 x {keys[0]}                      (a ratio)')
print('neither statement means anything for this column.')

The exception is a genuinely **ordered** column, where the numbers are the point. Here `class` really is
ordered — First, Second, Third — so a numeric code preserves real information rather than inventing it.

In [ ]:
ordered_candidates = [c for c in df.columns if isinstance(df[c].dtype, pd.CategoricalDtype)
                      and df[c].cat.ordered]
print('columns pandas already knows are ordered:', ordered_candidates)
print()

order = ['First', 'Second', 'Third']
col_ord = [c for c in encode_cols if set(df[c].astype(str).unique()) == set(order)]
if col_ord:
    c0 = col_ord[0]
    df[c0 + '_ordinal'] = df[c0].astype(str).map({v: i + 1 for i, v in enumerate(order)})
    print(f'{c0} -> {c0}_ordinal, because the order is real:')
    print(df.groupby(c0 + '_ordinal', observed=True)[target].agg(['size', 'mean']).round(3).to_string())

### 8.4 — Scaling: same data, different units

Three ways to put a column on a comparable scale, all computed by hand so there is no mystery in them:

| Method | Formula | What it is robust to |
|---|---|---|
| standardise (z-score) | `(x - mean) / std` | nothing — both parts move with outliers |
| min-max | `(x - min) / (max - min)` | nothing — both parts *are* the outliers |
| robust | `(x - median) / IQR` | outliers, by construction |

In [ ]:
scale_cols = [c for c in true_numeric if c in df.columns][:2]
print('scaling:', scale_cols)
print()

scaled = pd.DataFrame(index=df.index)
for c in scale_cols:
    x = df[c]
    scaled[c + '_z']      = (x - x.mean()) / x.std()
    scaled[c + '_minmax'] = (x - x.min()) / (x.max() - x.min())
    iqr_c = x.quantile(0.75) - x.quantile(0.25)
    scaled[c + '_robust'] = (x - x.median()) / iqr_c

print(scaled.describe().round(3).to_string())

### 8.5 — And what scaling does *not* do

This is the misconception worth a whole slide. Scaling moves and stretches a distribution; it never
changes its **shape**. A skewed column stays exactly as skewed after scaling — only the transform in
Part 5 changes that.

In [ ]:
c = scale_cols[0]
views = {
    'raw':          df[c],
    'standardised': scaled[c + '_z'],
    'min-max':      scaled[c + '_minmax'],
    'log1p':        np.log1p(df[c]),
}

fig, axs = plt.subplots(1, 4, figsize=(14, 2.9))
for ax, (name, v) in zip(axs, views.items()):
    ax.hist(v, bins=35, color='#BF8F00', edgecolor='white')
    ax.set_title(f'{name}\nskew {v.skew():.2f}', fontsize=10)
fig.suptitle(f'{c} — scaling changes the axis, not the shape', y=1.06, fontsize=12)
plt.tight_layout(); plt.show()

The first three pictures are the **same shape** with different numbers on the x-axis. Only the fourth is a
different distribution. Remember that the next time somebody suggests scaling to "fix" a skewed column.

### 8.6 — Put the analysis frame together

The numeric columns as they are, plus the encoded text columns. The reading-only columns — the bands —
stay out of it, because they are a coarser copy of something already in the table.

In [ ]:
numeric_part = df.select_dtypes(include='number')
analysis = pd.concat([numeric_part, encoded], axis=1)

print('rows           :', len(analysis))
print('columns in df  :', df.shape[1])
print('columns out    :', analysis.shape[1])
print('any missing?   :', bool(analysis.isna().any().any()))
print()
print(list(analysis.columns))

---
# Part 9 — What did we actually change?

Before and after, measured. This is the part everyone skips, and it is the part that catches the mistake
you made two hours ago.

### 9.1 — The diff, computed against the pristine copy

In [ ]:
original_cols = set(raw.columns)
final_cols    = set(df.columns)

print('columns dropped  :', sorted(original_cols - final_cols))
print('columns added    :', sorted(final_cols - original_cols))
print('columns kept     :', len(original_cols & final_cols))
print()
print('rows in          :', len(raw))
print('rows out         :', len(df))
print('rows lost        :', len(raw) - len(df))
print()
print('missing values in :', int(raw.isna().sum().sum()))
print('missing values out:', int(df.isna().sum().sum()))

Zero rows lost. That is not an accident — every decision in this notebook was chosen to keep rows, because
the rows we would have deleted were the interesting ones.

### 9.2 — The columns that were touched, before and after

For the numeric columns that both frames share, compare the summary statistics side by side. A column
whose mean moved a long way is a column whose imputation you should double-check.

In [ ]:
shared = [c for c in raw.columns if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]

compare_rows = []
for c in shared:
    compare_rows.append({
        'column': c,
        'mean_before': round(raw[c].mean(), 3),
        'mean_after':  round(df[c].mean(), 3),
        'std_before':  round(raw[c].std(), 3),
        'std_after':   round(df[c].std(), 3),
        'missing_before': int(raw[c].isna().sum()),
        'missing_after':  int(df[c].isna().sum()),
    })

changes = pd.DataFrame(compare_rows)
changes['mean_shift'] = (changes['mean_after'] - changes['mean_before']).round(3)
print(changes.to_string(index=False))

### 9.3 — And the same thing drawn

In [ ]:
touched = changes[changes['missing_before'] > 0]['column'].tolist()
print('columns whose values we filled:', touched)

if touched:
    fig, axs = plt.subplots(1, len(touched), figsize=(5 * len(touched), 3.2))
    for ax, c in zip(np.atleast_1d(axs), touched):
        ax.hist(raw[c].dropna(), bins=30, alpha=0.55, label='before (observed only)',
                color='#666666', edgecolor='white')
        ax.hist(df[c], bins=30, alpha=0.55, label='after (filled)',
                color='#548235', edgecolor='white')
        ax.set_title(c); ax.legend(fontsize=8)
    fig.suptitle('what filling the gaps did to the distribution', y=1.03, fontsize=12)
    plt.tight_layout(); plt.show()

---
# Part 10 — The one-page visual report

Everything so far was for you. This part is for somebody else — the four charts and four sentences you
would put in front of a person who is not going to read your code.

### 10.1 — The dashboard, built from the ranking

The four panels are `ranks.head(4)` — chosen by the statistics in Part 6, not by taste. Change the data
and the report redraws itself around whatever turns out to matter.

In [ ]:
report_cols = ranks['column'].head(4).tolist()
print('reporting on:', report_cols)

fig, axs = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axs.flat, report_cols):
    s = df[col]
    if pd.api.types.is_numeric_dtype(s) and s.nunique() > 10:
        sns.kdeplot(data=df, x=col, hue=target, fill=True, common_norm=False, ax=ax)
        ax.set_title(f'{col} — distribution by {target}')
    else:
        sns.barplot(x=col, y=target, data=df, ax=ax, errorbar=('ci', 95))
        ax.axhline(df[target].mean(), color='gray', linestyle='--', linewidth=1)
        ax.set_title(f'{col} — {target} rate (dashed = overall)')
        ax.tick_params(axis='x', rotation=20)

fig.suptitle('Titanic — the four columns that came top of both rankings', y=1.01, fontsize=14)
plt.tight_layout()
plt.savefig('titanic_report.png', dpi=150, bbox_inches='tight')
plt.show()

### 10.2 — Say what it shows, in numbers

A chart opens the conversation. The number closes it. Never present one without the other.

In [ ]:
for col in report_cols:
    s = df[col]
    print('=' * 60)
    if pd.api.types.is_numeric_dtype(s) and s.nunique() > 10:
        a, b = s[df[target] == 1], s[df[target] == 0]
        t, p = stats.ttest_ind(a, b, equal_var=False)
        print(f'{col}: group 1 mean {a.mean():.2f} vs group 0 mean {b.mean():.2f}   p = {p:.3g}')
    else:
        rate = df.groupby(col, observed=True)[target].agg(['size', 'mean']).round(3)
        print(f'{col}:')
        print(rate.to_string())

### 10.3 — ⚠️ The same data, drawn badly

Session 3's lesson, applied to this table one last time. Both panels below are honest arithmetic. Only one
of them is an honest picture.

In [ ]:
cat_for_chart = [c for c in df.columns if df[c].nunique() <= 4
                 and not pd.api.types.is_float_dtype(df[c])
                 and not str(df[c].dtype) == 'bool'][0]
rates = df.groupby(cat_for_chart, observed=True)[target].mean()

fig, axs = plt.subplots(1, 2, figsize=(11, 3.4))

axs[0].bar([str(i) for i in rates.index], rates.to_numpy(), color='#C0504D')
axs[0].set_ylim(rates.min() - 0.02, rates.max() + 0.02)
axs[0].set_title('truncated axis — the difference looks enormous')

bars = axs[1].bar([str(i) for i in rates.index], rates.to_numpy(), color='#548235')
axs[1].bar_label(bars, fmt='%.3f', padding=2)
axs[1].set_ylim(0, 1)
axs[1].set_title('axis from zero — the difference is what it is')

plt.tight_layout(); plt.show()
print(rates.round(3).to_string())

---
# Part 11 — Ship it: the cleaned table and its documentation

A cleaned CSV on its own is half a deliverable. The other half is a **data dictionary** — and it should be
generated from the frame, so it cannot drift out of date.

### 11.1 — Save the cleaned table

In [ ]:
import os

df.to_csv('titanic_clean.csv', index=False)
analysis.to_csv('titanic_analysis_matrix.csv', index=False)

print('titanic_clean.csv           :', os.path.getsize('titanic_clean.csv'), 'bytes',
      '  shape', df.shape)
print('titanic_analysis_matrix.csv :', os.path.getsize('titanic_analysis_matrix.csv'), 'bytes',
      '  shape', analysis.shape)

### 11.2 — Generate the data dictionary

One row per column: what type it is, how many values it has, whether we created it, and what we did to it.
Every field is computed.

In [ ]:
provenance = []
for c in df.columns:
    if c not in raw.columns:
        if c.endswith('_known'):        origin = 'created — indicator for a dropped column'
        elif c.endswith('_was_missing'):origin = 'created — flag for an imputed value'
        elif c.endswith('_log'):        origin = 'created — log1p transform'
        elif c.endswith('_band'):       origin = 'created — binned, for reading only'
        elif c.endswith('_ordinal'):    origin = 'created — ordered category as a number'
        elif c.endswith('_per_person'): origin = 'created — ratio'
        elif '_x_' in c:                origin = 'created — interaction of two columns'
        else:                            origin = 'created — derived feature'
    elif raw[c].isna().sum() > 0:       origin = f'original — {int(raw[c].isna().sum())} values imputed'
    else:                                origin = 'original — unchanged'
    provenance.append({
        'column': c,
        'dtype': str(df[c].dtype),
        'n_unique': int(df[c].nunique()),
        'example': df[c].iloc[0],
        'origin': origin,
    })

dictionary = pd.DataFrame(provenance)
dictionary.to_csv('titanic_data_dictionary.csv', index=False)
print(dictionary.to_string(index=False))

### 11.3 — A written summary, generated from the run

Every number below is computed at the moment it is printed. A summary typed by hand goes stale the first
time you change a threshold; this one cannot.

In [ ]:
lines = [
    'TITANIC — PREPROCESSING & VISUALIZATION SUMMARY',
    '=' * 52,
    f'rows in             : {len(raw)}',
    f'rows out            : {len(df)}',
    f'columns in          : {raw.shape[1]}',
    f'columns dropped     : {len(to_drop) + len([c for c in plan if plan[c] == "indicator + drop"])}',
    f'columns engineered  : {len([c for c in df.columns if c not in raw.columns])}',
    f'columns out         : {df.shape[1]}',
    f'analysis matrix     : {analysis.shape[1]} columns after encoding',
    '',
    f'missing values in   : {int(raw.isna().sum().sum())}',
    f'missing values out  : {int(df.isna().sum().sum())}',
    f'listwise deletion would have cost : {1 - len(raw.dropna())/len(raw):.1%} of the rows',
    '',
    f'strongest relationship : {ranks.iloc[0]["column"]}',
    f'most skewed column     : {outliers.iloc[0]["column"]} (skew {outliers.iloc[0]["skew"]})',
]
report = '\n'.join(lines)
print(report)

with open('titanic_summary.txt', 'w') as f:
    f.write(report)

---
# ✍️ Your turn — four exercises

### T.1 — Make the duplicate detector stricter

The one-to-one test in Part 3.2 only looks at columns with 20 or fewer distinct values. Rewrite it so it
also handles a **numeric** pair by checking whether one column is an exact linear function of the other
(hint: `.corr()` of exactly 1.0, or `nunique()` of the ratio being 1). Test it by adding a column that is
`fare * 2` and confirming your code finds it.

In [ ]:
# your code here


### T.2 — A different imputation, measured

Fill the numeric column three more ways: with the overall mean, with a forward fill, and with a random
draw from the observed values. For each one report the mean, the standard deviation, and draw the
distribution. Which one keeps the shape closest to the observed data, and why?

In [ ]:
# your code here


### T.3 — A chart that lies

Take any relationship in this table and produce **two** charts of it: one that overstates it and one that
is honest. You may only change chart settings — axis limits, chart type, ordering, colour — never the
numbers. Then write one sentence on which setting did the damage.

In [ ]:
# your code here


### T.4 — The whole notebook, on a different dataset

Load `sns.load_dataset('penguins')`, set `target = 'species'`, and run Parts 1, 2, 3 and 4 **unchanged**.
Everything is derived, so it should work. Write down anything that breaks — that is where a hidden
assumption was hiding.

In [ ]:
# your code here


---
# Recap

| Step | The line |
|---|---|
| Profile | build a frame of dtype / missing / unique — never read `info()` by eye |
| Type the columns | `select_dtypes` + a `nunique()` threshold, never a typed list |
| See the whole table | missingness matrix, then one histogram grid and one count grid, from the derived lists |
| Duplicate columns | groupby-nunique one-to-one test, proved with a crosstab |
| Missing mechanism | t-test / chi-square of *missing vs present* on every other column |
| Heavy missing | keep a `_known` indicator, drop the column |
| Numeric missing | group-wise median + a `_was_missing` flag, and draw the before and after |
| Outliers | IQR per column, then per group; `log1p` for money; keep the rows |
| Relationships | effect size **and** correlation ratio — where they disagree is the interesting part |
| Features | derive from a rule, then defend with a number and a picture |
| Encoding | `get_dummies` with `drop_first`; never number an unordered category |
| Scaling | changes the axis, not the shape |
| Report | four charts chosen by the ranking, four sentences with the numbers |
| Ship | the cleaned CSV **and** a generated data dictionary |

**The five sentences**

1. Derive every list; type none of them.
2. Print the evidence before the decision, in the same cell.
3. Look at the table before you clean it, and look again after.
4. A rare value is data; only an impossible value is an error.
5. The deliverable is the clean table plus the document that explains it.

---
# Homework

1. Run Parts 1 to 6 on a dataset of your own choosing — `penguins`, `diamonds`, `tips`, or any CSV. Do not
   edit any of the derivation code. Submit the profile table, the missingness figure, and one paragraph on
   what the code got wrong about your data.
2. Take the imputation in Part 4 and change the grouping keys. Report the effect on the mean and the
   standard deviation, and include the before-and-after histogram.
3. Produce a one-page visual report of your dataset: four charts and four sentences, each sentence
   containing a number that comes from the code.

Number 3 is the one that matters most in a real job — nobody reads the notebook, everybody reads the page.